# Nes2Net: tái hiện paper (Kaggle · Colab · local)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trinhphamtrunghieu/nes2net-fake-voice-detection/blob/main/notebooks/nes2net.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/trinhphamtrunghieu/nes2net-fake-voice-detection/blob/main/notebooks/nes2net.ipynb)

**Paper:** *Nes2Net: A Lightweight Nested Architecture for Foundation Model Driven Speech Anti-spoofing*
(Liu, Truong, Das, Lee, Li, IEEE T-IFS 2025, [arXiv:2504.05657](https://arxiv.org/abs/2504.05657)).

Một notebook cho mọi nền tảng: notebook tự nhận Kaggle / Colab / local và chọn chỗ lưu (`/kaggle/working`,
Google Drive, `./work`). Code nằm trong package `nes2net/`; notebook chỉ chọn **preset** và gọi các hàm của nó.

| Preset | Chạy gì | Thời gian (T4) |
|---|---|---|
| `smoke` | tập con rất nhỏ của mọi phần, chỉ để kiểm tra pipeline | ~30 phút |
| `default` | **A** checkpoint chính thức trên tập con · **C** ablation · **C2** chọn layer SSL | ~3–4 giờ |
| `full_eval` | **A** trên toàn bộ LA19 / ITW / 2021 LA / 2021 DF · **C** · **C2** | ~6–8 giờ |
| `paper_train` | **B** train đúng công thức paper (lr 2.5e-7, bs 12, 100 epoch, RawBoost 4), resume qua nhiều phiên | ~80–100 giờ |

**Chuẩn bị:** GPU (Kaggle: T4, Internet **On**; Colab: *Runtime → Change runtime type → GPU*, Pro nên chọn A100/L4).
Trên Colab, ô Setup xin quyền mount Google Drive để lưu kết quả và trạng thái resume.
Dữ liệu tải từ Hugging Face Hub (`Bisher/ASVspoof_2019_LA`, `SpeechAntiSpoofingBenchmarks/*`).

In [1]:
# ---------------- Setup: clone/cập nhật repo, cài thư viện ----------------
import os, subprocess, sys
REPO_URL = 'https://github.com/trinhphamtrunghieu/nes2net-fake-voice-detection.git'
REPO_REF = 'feat/vi-zeroshot'   # ghim một commit/tag (vd '154b3b4') cho các lần train dài để code không đổi giữa các phiên
IN_COLAB, IN_KAGGLE = 'google.colab' in sys.modules, os.path.isdir('/kaggle/working')
REPO_DIR = ('/content/nes2net-fake-voice-detection' if IN_COLAB else
            '/tmp/nes2net-fake-voice-detection' if IN_KAGGLE else os.path.abspath('nes2net-fake-voice-detection'))
if 'nes2net' in sys.modules:
    print('!! nes2net đã được import trong runtime này: Runtime → Restart session rồi chạy lại để nạp code mới')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_REF], check=True)
if REPO_REF == 'main':
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q', 'origin', 'main'], check=True)
print('code @', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(REPO_DIR, 'requirements.txt')], check=True)
sys.path.insert(0, REPO_DIR)

code @ cc8eaab save2



In [2]:
# ---------------- Config ----------------
PRESET = 'paper_train'   # 'smoke' | 'default' | 'full_eval' | 'paper_train'
OVERRIDES = {}       # ghi đè bất kỳ khoá nào, vd {'B': {'lr': 2e-6}, 'run': {'L': False}}
USE_DRIVE = True     # Colab: lưu WORK lên Google Drive

from nes2net import env as nenv, presets, run
from IPython.display import display
ENV = nenv.setup(use_drive=USE_DRIVE)
cfg = presets.get(PRESET, ENV.platform, OVERRIDES)
print(ENV); presets.show(cfg)

Mounted at /content/drive
platform=colab device=cuda workers=12
  WORK=/content/drive/MyDrive/nes2net
  TMP =/content/nes2net_tmp
preset=paper_train platform=colab parts=['sanity', 'B']
  B: lr=2.5e-07 bs=12 epochs=100 algo=4 dev=asv19:validation time_budget_h=11.5 keep_best_k=3


"preset=paper_train platform=colab parts=['sanity', 'B']\n  B: lr=2.5e-07 bs=12 epochs=100 algo=4 dev=asv19:validation time_budget_h=11.5 keep_best_k=3\n"

## 0. Kiểm tra nhanh: back-end khớp bit-for-bit với code gốc, số tham số ≈ 511k

In [3]:
if cfg['run']['sanity']:
    display(run.sanity(ENV))

KeyboardInterrupt: 

## A. Checkpoint chính thức → đánh giá
`load_official_checkpoint` chuyển key fairseq sang HF `Wav2Vec2Model` (không cần fairseq) và nạp **strict**.
Giao thức: cửa sổ 4 s đầu như code gốc; ITW chấm thêm cả câu. ASVspoof 2021: script chấm chính thức chỉ tính
trial `phase == eval` (con số của paper); Arena tính **toàn bộ** trial. Preset `full_eval` báo cả hai, kèm EER theo codec/vocoder.

In [ ]:
if cfg['run']['A']:
    RA = run.eval_official(cfg, ENV)
    display(run.results_table(RA))
if cfg['run']['A_full']:
    RAF = run.eval_official_full(cfg, ENV)       # chạy lại được: shard đã chấm sẽ bỏ qua
    display(run.results_table(RAF))
    for k, v in RAF.items():
        if 'phase_eval_breakdown_eer' in v:
            print(k, {a: round(e, 2) for a, e in v['phase_eval_breakdown_eer'].items()})

## B. Train end-to-end
XLS-R 300M fine-tune + Nes2Net-X `[8,8]`, SE 1, dilation 2, mean pooling, Adam (wd 1e-4), weighted CE `[0.1, 0.9]`,
RawBoost algo 4, cửa sổ 4 s đầu; lưu checkpoint khi val loss (có/không aug) giảm.

**Resume:** trạng thái ghi vào `WORK/runB/last_state.pth` mỗi `save_every_steps` bước và cuối mỗi epoch; mỗi phiên dừng
an toàn sau `time_budget_h`. Bị ngắt hoặc hết giờ → chạy lại notebook với cùng preset. Trên Kaggle, thêm output của
lần chạy trước làm input của kernel: `runB/` được copy về tự động. Khi đủ epoch: `OVERRIDES = {'run': {'B': False, 'B_eval': True}}`.

In [ ]:
if cfg['run']['B']:
    HIST = run.train(cfg, ENV)
if cfg['run']['B_eval']:
    display(run.eval_trained(cfg, ENV))

### Log của phần B
`WORK/runB/train.log` (dạng chữ), `metrics.jsonl` (JSON từng sự kiện), `tb/` (TensorBoard). Cứ `log_every` bước một
dòng gồm loss, lr, utt/s, % thời gian chờ DataLoader, GPU memory, ETA; cuối epoch: val loss và val EER.

In [ ]:
SHOW_TB = False
if cfg['run']['B'] or os.path.exists(os.path.join(ENV.work, cfg['B']['out'], 'metrics.jsonl')):
    run.plot_train_logs(os.path.join(ENV.work, cfg['B']['out']), show_tb=SHOW_TB)

## C. Ablation back-end trên đặc trưng XLS-R đóng băng
Đặc trưng XLS-R (pre-trained, không fine-tune) của cửa sổ 4 s được cache fp16 một lần; mỗi back-end train vài phút.
EER tuyệt đối cao hơn paper, chỉ so sánh **tương đối** giữa các biến thể (mean ± std qua seed).
Nhóm A0–A5: baseline, tỉ lệ lồng, SE, pooling, dilation/kernel, bỏ kết nối tích luỹ (xem `docs/ablation_and_gaps.md`).

In [ ]:
if cfg['run']['C']:
    from nes2net.ablation import summarize
    DFC = run.ablation(cfg, ENV)
    display(summarize(DFC))
    run.plot_ablation(DFC, ENV)

### C2. Chọn layer SSL (research gap G1): từng layer so với tổng có trọng số học được

In [ ]:
if cfg['run']['L']:
    DFL = run.layer_study(cfg, ENV)
    display(DFL.groupby('layers')[['dev_eer', 'itw_eer']].agg(['mean', 'std']).round(2))

## Kết quả trong `WORK`
`partA_results.json` · `partA_full_results.json` + `A_full/` · `runB/` · `partB_results.json` ·
`partC_ablation.csv/.png` · `partC_layers.csv`. Dòng lệnh tương đương: `python -m nes2net.run --preset <tên>`.